# SBERT / Contenido (H3) - con protocolo comun `recsys_protocol`

Notebook de **H3** (carpeta `H3/`). Version del notebook de contenido (CB-SBERT,
CB-TF-IDF, ALS, Most Popular) que adopta `recsys_protocol.py`. **No toca H2:** el
original queda intacto en `../H2/H2_sbert_content.ipynb`.

**Cambios vs H2** (para *aislar* el efecto del protocolo):
- `set_global_seed()` (FIX #3) y muestreo `sample_users()` determinista e
  independiente del orden (FIX #2): el eval pool es ahora **el mismo que en
  `../H3/BPR.ipynb`** -> las tablas entre notebooks son comparables.
- k-core, split (`build_splits`) y metricas vienen del **modulo** (unica fuente
  de verdad), verificadas numericamente identicas a H2.
- **Novelty estandarizada** (denominador = nro interacciones positivas en train,
  ignora items no vistos) = misma convencion para todos los modelos. Ojo: H2 BPR
  usaba otra escala (nro usuarios), asi que los valores de Novelty no son
  directamente comparables con los de H2.

**Runtime sugerido en Colab:** GPU **T4** + High-RAM (el encoding SBERT se
acelera en GPU y la T4 basta; el resto del costo es cargar el CSV de 41M filas).


# Entrega H2 — Recomendador de contenido semántico (SentenceBERT)
## IIC3633 Sistemas Recomendadores · Benjamín Suazo y Pablo Lagos

Este notebook es **autocontenido**: descarga los datos, reproduce el catálogo activo
(k-core) y el split *leave-one-out* temporal de H1, construye **embeddings de texto con
SentenceBERT** y evalúa un recomendador de **contenido semántico**, comparándolo contra
los baselines de H1 (Most Popular, Content-Based TF-IDF, ALS).

Responde a tres puntos del feedback de H1 (87/100):
- **Embeddings de texto** prometidos en la planificación Midterm (reemplazo semántico del TF-IDF de tags).
- **Métricas de novedad/diversidad** (Coverage, Intra-List Diversity, Novelty) además de ranking.
- **Cortes @5 / @10 / @20** y **desagregación por nivel de actividad del usuario**.

### Cómo correrlo
1. Entorno de ejecución → **GPU** (Entorno de ejecución ▸ Cambiar tipo de entorno ▸ T4 GPU). Idealmente *High-RAM*.
2. Ejecutar todo (Entorno de ejecución ▸ Ejecutar todo). La 1ª vez `kagglehub` pedirá login de Kaggle.
3. Tiempo estimado: ~10–15 min (la mayor parte es leer `recommendations.csv`, ~1.4 GB).

> El catálogo del proyecto y los cortes (k-core ≥5 reseñas/usuario, ≥20/juego; submuestra 10% de usuarios; eval ≤2.000 usuarios) son **idénticos a H1** para que los resultados sean comparables.

## 0. Instalación e importaciones

In [ ]:
!pip install -q kagglehub sentence-transformers implicit

In [ ]:
import os, sys, json, shutil, zipfile, time, warnings, glob, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import kagglehub
warnings.filterwarnings('ignore')

# -- Protocolo experimental comun: lo trae el repo de la entrega en H3/ --
REPO_URL = 'https://github.com/Benjaa7/Proyecto-RecSys.git'   # <-- ajusta si tu repo cambia
REPO_DIR = '/content/Proyecto-RecSys'
def _locate_protocol():
    # Repo PRIMERO: clonar si falta; si ya esta clonado, traer la ULTIMA version
    # (fetch + reset --hard). Esto evita el bug de usar un clon viejo cacheado de
    # una sesion anterior (un 'pull' tras encontrar copia local no se ejecutaba).
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', '--depth', '1', 'origin'], check=False)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '-q', 'FETCH_HEAD'], check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '-q', REPO_URL, REPO_DIR], check=False)
    h3 = os.path.join(REPO_DIR, 'H3')
    if os.path.exists(os.path.join(h3, 'recsys_protocol.py')):
        return h3
    # Fallback: copia local suelta (subida a mano), solo si el repo no esta disponible.
    for c in ['/content', '.', '..'] + sorted(glob.glob('/content/*')) + sorted(glob.glob('/content/drive/MyDrive/*')):
        if c and os.path.exists(os.path.join(c, 'recsys_protocol.py')):
            return c
    return None
_root = _locate_protocol()
if _root is None:
    raise ModuleNotFoundError("No encuentro recsys_protocol.py. Revisa que este en H3/ del repo.")
sys.path.insert(0, _root)
# Re-import fresco: si el modulo ya estaba en memoria (re-correr la celda sin
# reiniciar el runtime), descartarlo para tomar la version recien traida.
for _m in [m for m in list(sys.modules) if m == 'recsys_protocol' or m.startswith('recsys_protocol.')]:
    del sys.modules[_m]
print('recsys_protocol desde:', _root)

import recsys_protocol as proto
from recsys_protocol import (SEED, ProtocolConfig, set_global_seed, sample_users,
    iterative_k_core, build_splits, positives, popularity_counts,
    evaluate_at_ks, ndcg_at_k, reproducibility_note)

set_global_seed()  # FIX #3: seed unica de proyecto

# -- Configuracion (igual a H1/H2 para comparabilidad) --
SLUG      = "antonkozyriev/game-recommendations-on-steam"
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # rapido; all-mpnet-base-v2 = mas calidad
TEXT_MODE = "combined"                                  # "combined" o "description"
cfg = ProtocolConfig(frac_eval=0.10, frac_train=0.10, min_user=5, min_game=20,
                     max_eval_users=2000, ks=(5, 10, 20),
                     user_col='user_id', item_col='app_id', time_col='date',
                     pos_col='is_recommended')
# alias para celdas sin tocar
RANDOM_STATE = SEED
KS = tuple(cfg.ks)
MIN_USER_REVIEWS, MIN_GAME_REVIEWS = cfg.min_user, cfg.min_game
SAMPLE_FRAC, MAX_EVAL_USERS = cfg.frac_eval, cfg.max_eval_users
print(reproducibility_note(cfg, 'Kozyriev (SBERT piloto)'))


## 1. Descarga robusta de datos

`kagglehub` a veces entrega los archivos comprimidos en ZIP y `games.csv` de este
dataset no es UTF-8; los helpers de abajo lo resuelven (mismo criterio del análisis de
cobertura de H2).

In [ ]:
def _find_file(root: Path, *names):
    cand = {n.lower() for n in names}
    for p in Path(root).rglob('*'):
        if p.is_file() and p.name.lower() in cand:
            return p
    raise FileNotFoundError(f'No encontré {names} bajo {root}')

def _maybe_unzip(p: Path, *wanted):
    with open(p, 'rb') as f:
        if f.read(4) != b'PK\x03\x04':
            return p
    ext = p.parent / (p.stem + '_extracted'); ext.mkdir(exist_ok=True)
    with zipfile.ZipFile(p) as z:
        names = z.namelist(); target = None
        for w in wanted:
            target = next((n for n in names if Path(n).name.lower() == w.lower()), None)
            if target: break
        if target is None:
            target = next((n for n in names if n.lower().endswith(('.csv', '.json'))), names[0])
        out = ext / Path(target).name
        if not out.exists() or out.stat().st_size == 0:
            with z.open(target) as s, open(out, 'wb') as d: shutil.copyfileobj(s, d)
        return out

def fetch_file(slug, *filenames):
    for fn in filenames:
        try:
            p = Path(kagglehub.dataset_download(slug, path=fn))
            p = p if p.is_file() else _find_file(p, fn)
            return _maybe_unzip(p, *filenames)
        except Exception:
            continue
    return _find_file(Path(kagglehub.dataset_download(slug)), *filenames)

def read_csv_robust(path, **kw):
    for enc in ('utf-8', 'utf-8-sig', 'cp1252', 'latin-1'):
        try:
            return pd.read_csv(path, encoding=enc, **kw)
        except (UnicodeDecodeError, UnicodeError):
            continue
    return pd.read_csv(path, encoding='latin-1', encoding_errors='replace', **kw)

t0 = time.time()
games_csv = fetch_file(SLUG, 'games.csv')
meta_json = fetch_file(SLUG, 'games_metadata.json')
recs_csv  = fetch_file(SLUG, 'recommendations.csv')
print(f'Descargado en {time.time()-t0:.0f}s')
print(' games.csv   ->', games_csv)
print(' metadata    ->', meta_json)
print(' recs.csv    ->', recs_csv)

## 2. Carga de interacciones y filtrado k-core

Se carga `recommendations.csv` completo (~41M filas) en *chunks* con tipos optimizados,
se deduplica por par (user, juego) y se aplica el **k-core iterativo** (≥5 reseñas por
usuario, ≥20 por juego). Debe converger en **~22.676 juegos** (validando la reproducción
del catálogo activo de H1).

In [ ]:
t0 = time.time()
parts = []
for chunk in pd.read_csv(recs_csv,
        usecols=['app_id','user_id','hours','is_recommended','date'],
        chunksize=2_000_000,
        dtype={'app_id':'int32','user_id':'int32','hours':'float32','is_recommended':'bool'}):
    parts.append(chunk)   # 'date' se deja como string aquí; se parsea solo en la submuestra (mucho más rápido)
recs_full = pd.concat(parts, ignore_index=True); del parts
print(f'Interacciones cargadas: {len(recs_full):,} en {time.time()-t0:.0f}s')

# Una sola observación por par (user, app) — fiel a H1
recs_full = recs_full.sort_values('date').drop_duplicates(['user_id','app_id'], keep='last', ignore_index=True)
print(f'Pares únicos (user, juego): {len(recs_full):,}')

In [ ]:
# k-core iterativo (modulo: identico al de H2)
print('k-core iterativo...')
recs_filtered = iterative_k_core(recs_full, MIN_USER_REVIEWS, MIN_GAME_REVIEWS, verbose=True)
del recs_full
active_games = set(recs_filtered['app_id'].unique())
print(f'\nCatalogo activo: {len(active_games):,} juegos | interacciones: {len(recs_filtered):,}')


## 3. Submuestra de usuarios y split *leave-one-out* temporal

Submuestra del 10% de usuarios (como H1) y, por usuario, su interacción **más reciente**
es el ítem de test; el resto es entrenamiento. Solo se evalúan usuarios cuyo ítem de test
es **positivo** (`is_recommended=True`).

In [ ]:
# Senal de confianza: log1p(hours) en positivos, 0 en negativos
recs_filtered = recs_filtered.copy()
recs_filtered['confidence'] = np.where(recs_filtered['is_recommended'],
                                       np.log1p(recs_filtered['hours'].clip(0)), 0.0)

# -- Diagnostico FIX #2 (sobre el universo COMPLETO post-k-core) --
_u = recs_filtered['user_id'].to_numpy()
_us = np.random.default_rng(0).permutation(_u)
_na, _nb = set(sample_users(_u, cfg.frac_eval, cfg.seed)), set(sample_users(_us, cfg.frac_eval, cfg.seed))
_oa = set(pd.Series(pd.unique(_u)).sample(frac=cfg.frac_eval, random_state=cfg.seed))
_ob = set(pd.Series(pd.unique(_us)).sample(frac=cfg.frac_eval, random_state=cfg.seed))
print(f'[FIX#2] nuevo -> pools iguales bajo reordenamiento: {_na == _nb} (|pool|={len(_na):,})')
print(f'[FIX#2] viejo -> pools iguales: {_oa == _ob} | overlap={len(_oa & _ob)/max(len(_oa),1)*100:.1f}%')

# Muestreo DETERMINISTA + split LOO temporal con politica de pools (modulo).
# Usa la MISMA seed/fraccion que BPR -> el eval pool es identico -> tablas comparables.
sp = build_splits(recs_filtered, cfg)
train_df             = sp.train_df
test_pos             = sp.test_pos
test_item_per_user   = sp.test_item_per_user
train_items_per_user = sp.train_items_per_user
eval_users           = sp.eval_users
print(sp.summary())


In [ ]:
# Catalogo recomendable = items presentes en train (consistente con BPR/CPGRec)
ALL_ITEMS = set(train_df['app_id'].unique())
item_list = sorted(ALL_ITEMS)
item_idx  = {a: i for i, a in enumerate(item_list)}
n_catalog = len(item_list)
print(f'Catalogo recomendable: {n_catalog:,} juegos | usuarios evaluados: {len(eval_users):,}')


## 4. Texto por juego (título + descripción + género + tags)

Se arma el texto a codificar desde `games.csv` (títulos) y `games_metadata.json`
(descripciones + tags). Con `TEXT_MODE="description"` se codifica solo título+descripción
(ablación "semántica pura" vs TF-IDF de tags).

In [ ]:
# Títulos
games = read_csv_robust(games_csv, usecols=['app_id','title'])
games['app_id'] = pd.to_numeric(games['app_id'], errors='coerce')
title_map = dict(zip(games['app_id'].dropna().astype('int64'),
                     games['title'].astype(str)))

# Tags + descripciones (JSONL)
desc_map, tags_map = {}, {}
with open(meta_json, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: rec = json.loads(line)
        except json.JSONDecodeError: continue
        aid = rec.get('app_id')
        if aid is None: continue
        aid = int(aid)
        desc_map[aid] = (rec.get('description') or '').strip()
        tags_map[aid] = rec.get('tags') or []

def build_text(aid):
    title = title_map.get(aid, '').strip()
    d = desc_map.get(aid, '')
    if TEXT_MODE == 'description':
        return f'{title}. {d}'.strip('. ').strip() or (title or 'unknown game')
    tags = ', '.join(tags_map.get(aid, []))
    parts = [title]
    if d: parts.append(d)
    if tags: parts.append(f'Tags: {tags}')
    txt = '. '.join(p for p in parts if p).strip()
    return txt or (title or 'unknown game')

texts = [build_text(a) for a in item_list]
pct_desc = 100*np.mean([bool(desc_map.get(a)) for a in item_list])
print(f'Textos construidos: {len(texts):,} | con descripción: {pct_desc:.1f}%')
print('Ejemplo:\n ', texts[0][:300])

## 5. Embeddings con SentenceBERT

Se codifica cada juego en un vector denso L2-normalizado (coseno = producto punto).

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', device)
emb_model = SentenceTransformer(EMB_MODEL, device=device)

t0 = time.time()
emb = emb_model.encode(texts, batch_size=128, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True).astype('float32')
print(f'\nEmbeddings: {emb.shape[0]:,} juegos x {emb.shape[1]} dims en {time.time()-t0:.0f}s')

## 6. Métricas

Ranking (Precision/Recall/NDCG) en varios cortes + diversidad/novedad:
- **Coverage@k**: fracción del catálogo que aparece en algún top-k (cobertura de cola).
- **ILD@k** (Intra-List Diversity): 1 − coseno promedio entre pares del top-k (en el espacio de embeddings).
- **Novelty@k**: −log2(popularidad) promedio (alto = recomienda ítems menos populares).

Son agnósticas al modelo: reciben `{user_id: [items rankeados]}` de cualquier recomendador.

In [ ]:
# Las metricas NO se redefinen aqui: vienen de recsys_protocol (importadas en
# el setup). Verificado numericamente identicas a las de H2 para este pipeline
# (evaluate_at_ks, Coverage, ILD, Novelty). Novelty usa la convencion canonica
# del proyecto (denominador = nro interacciones positivas en train, ignora no vistos).
_ndcg = ndcg_at_k                               # alias para la desagregacion por actividad
popularity  = popularity_counts(train_df, cfg)  # interacciones positivas por item en train
total_inter = sum(popularity.values()) or 1

def diversity_metrics(recs, k=10):
    # enlaza el bundler del modulo a los datos de este notebook (emb ya normalizado)
    return proto.diversity_metrics(recs, popularity=popularity, total=total_inter,
                                   item_to_idx=item_idx, factors=emb,
                                   n_catalog=n_catalog, k=k, assume_normalized=True)
print('Metricas desde recsys_protocol (evaluate_at_ks, diversity_metrics, _ndcg)')


## 7. Modelos y evaluación

Se evalúan cuatro modelos bajo el mismo protocolo: **Most Popular**, **Content-Based
TF-IDF (tags)**, **Content-Based SBERT (semántico)** y **ALS**. Los dos de contenido
comparten el mismo motor (perfil = promedio de vectores ponderado por confianza →
coseno), cambiando solo la representación (TF-IDF vs embeddings).

In [ ]:
TOP_N = max(KS)

# ── Most Popular (también sirve de fallback) ──
popular_list = list(train_df[train_df['is_recommended']].groupby('app_id').size()
                    .sort_values(ascending=False).index)
recs_pop = {}
for u in eval_users:
    seen = train_items_per_user.get(u, set())
    recs_pop[u] = [i for i in popular_list if i not in seen][:TOP_N]
print('Most Popular listo')

In [ ]:
from sklearn.preprocessing import normalize

def content_recommend(M, top_n=TOP_N):
    # M: matriz (n_catalog, d) con filas L2-normalizadas, alineada con item_list
    train_pos = train_df[train_df['is_recommended']]
    grouped = {u: g for u, g in train_pos[['user_id','app_id','confidence']].groupby('user_id')}
    recs = {}; n_fb = 0
    for u in eval_users:
        seen = train_items_per_user.get(u, set())
        hist = grouped.get(u)
        rows = hist[hist['app_id'].isin(item_idx)] if hist is not None else None
        if rows is None or len(rows) == 0:
            recs[u] = [i for i in popular_list if i not in seen][:top_n]; n_fb += 1; continue
        idxs = [item_idx[a] for a in rows['app_id']]
        w = rows['confidence'].to_numpy(np.float32); w = np.where(w==0, 1e-6, w); w = w/w.sum()
        prof = w @ M[idxs]; nrm = np.linalg.norm(prof)
        if nrm > 0: prof = prof/nrm
        scores = M @ prof
        for s in seen:
            j = item_idx.get(s)
            if j is not None: scores[j] = -np.inf
        nt = min(top_n, len(scores))
        top = np.argpartition(-scores, nt-1)[:nt]; top = top[np.argsort(-scores[top])]
        recs[u] = [item_list[i] for i in top]
    return recs, n_fb

# ── Content-Based TF-IDF (tags) ──
from sklearn.feature_extraction.text import TfidfVectorizer
tag_corpus = [' '.join(tags_map.get(a, [])) for a in item_list]
tfidf = TfidfVectorizer(max_features=3000, min_df=2).fit_transform(tag_corpus)
M_tfidf = normalize(tfidf.toarray().astype(np.float32))
recs_tfidf, fb1 = content_recommend(M_tfidf)
print(f'TF-IDF listo (fallback Most Popular: {fb1})')

# ── Content-Based SBERT (semántico) ──
recs_sbert, fb2 = content_recommend(emb)   # emb ya está normalizado
print(f'SBERT listo (fallback Most Popular: {fb2})')

In [ ]:
# ── ALS (filtrado colaborativo, feedback implícito) ──
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix

user_list = sorted(train_df['user_id'].unique())
user_idx = {u: i for i, u in enumerate(user_list)}
tp = train_df[train_df['is_recommended']]
tp = tp[tp['user_id'].isin(user_idx) & tp['app_id'].isin(item_idx)]
ALPHA = 40
rows = tp['user_id'].map(user_idx).to_numpy()
cols = tp['app_id'].map(item_idx).to_numpy()
data = (1.0 + ALPHA*tp['confidence'].to_numpy()).astype('float32')
user_item = csr_matrix((data, (rows, cols)), shape=(len(user_list), n_catalog))

als = AlternatingLeastSquares(factors=64, regularization=0.1, iterations=15, random_state=RANDOM_STATE)
als.fit(user_item, show_progress=True)

recs_als = {}
for u in eval_users:
    seen = train_items_per_user.get(u, set())
    if u not in user_idx:
        recs_als[u] = [i for i in popular_list if i not in seen][:TOP_N]; continue
    ui = user_idx[u]
    ids, _ = als.recommend(ui, user_item[ui], N=TOP_N + len(seen), filter_already_liked_items=True)
    rec = [item_list[j] for j in ids if item_list[j] not in seen][:TOP_N]
    recs_als[u] = rec
print('ALS listo')

## 8. Comparación de resultados

In [ ]:
all_recs = {
    'Most Popular':       recs_pop,
    'CB-TF-IDF (tags)':   recs_tfidf,
    'CB-SBERT (texto)':   recs_sbert,
    'ALS (Colaborativo)': recs_als,
}

rows = {}
for name, recs in all_recs.items():
    m = evaluate_at_ks(recs, test_item_per_user)
    m.update(diversity_metrics(recs, k=10))
    rows[name] = m

results_df = pd.DataFrame(rows).T
cols_order = [f'{x}@{k}' for k in KS for x in ('Precision','Recall','NDCG')] + \
             ['Coverage@10','ILD@10','Novelty@10']
results_df = results_df[cols_order].astype(float).round(4)
results_df.to_csv('resultados_h2_sbert.csv')
print('Guardado: resultados_h2_sbert.csv')
results_df

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, k in zip(axes, KS):
    v = results_df[f'NDCG@{k}']
    bars = ax.bar(v.index, v.values, color=['#E67E22','#3498DB','#27AE60','#8E44AD'], edgecolor='black')
    ax.set_title(f'NDCG@{k}'); ax.tick_params(axis='x', rotation=25)
    for b, val in zip(bars, v.values):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.0005, f'{val:.4f}', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig('ndcg_h2_sbert.png', dpi=120, bbox_inches='tight'); plt.show()

## 9. Desagregación por nivel de actividad del usuario

El feedback pidió entender **cuándo cada modelo aporta valor**. Se agrupan los usuarios
evaluados según cuántas interacciones tienen en train (proxy de cola larga vs. usuarios
activos) y se reporta NDCG@10 por grupo. La hipótesis: el contenido (SBERT) debería
ayudar más a usuarios con poco historial, donde la señal colaborativa es débil.

In [ ]:
n_hist = {u: len(train_items_per_user.get(u, set())) for u in eval_users}
def bucket(n):
    if n <= 5:  return '1) 2-5'
    if n <= 10: return '2) 6-10'
    if n <= 20: return '3) 11-20'
    return '4) 21+'
user_bucket = {u: bucket(n_hist[u]) for u in eval_users}

records = []
for name, recs in all_recs.items():
    per_bucket = {}
    for u, rec in recs.items():
        if u not in test_item_per_user: continue
        b = user_bucket[u]
        per_bucket.setdefault(b, []).append(_ndcg(rec, {test_item_per_user[u]}, 10))
    for b, vals in per_bucket.items():
        records.append({'modelo': name, 'grupo': b, 'NDCG@10': np.mean(vals), 'n_users': len(vals)})

disagg = pd.DataFrame(records)
pivot = disagg.pivot(index='grupo', columns='modelo', values='NDCG@10').round(4).sort_index()
counts = disagg.groupby('grupo')['n_users'].first()
pivot['n_users'] = counts
pivot.to_csv('desagregacion_actividad_h2.csv')
print('Usuarios por grupo de actividad (nº interacciones en train):')
pivot

In [ ]:
# Descargar resultados (Colab)
try:
    from google.colab import files
    for f in ['resultados_h2_sbert.csv', 'desagregacion_actividad_h2.csv', 'ndcg_h2_sbert.png']:
        if os.path.exists(f): files.download(f)
except Exception as e:
    print('Archivos guardados en el entorno:', e)

## 10. Notas para el informe H2

- **Pregunta central de esta parte:** ¿el texto semántico (SBERT) supera a los tags
  (TF-IDF) como señal de contenido? Comparar la fila `CB-SBERT` vs `CB-TF-IDF`.
- **Ablación:** volver a correr con `TEXT_MODE = "description"` (celda de config) para
  aislar el aporte de la descripción semántica sobre los tags.
- **Diversidad/novedad:** reportar Coverage/ILD/Novelty además del ranking (lo pidió el feedback).
- **Desagregación:** discutir si SBERT ayuda más en usuarios de poco historial (cola larga).
- Estos cuatro modelos son baselines/contenido; el **modelo avanzado CPGRec+** y **BPR**
  van en sus propios desarrollos y se integran con estos embeddings en el híbrido.